# Visualizations for Influenza Outbreak Event Prediction via Twitter Data

Data sourced from https://archive.ics.uci.edu/dataset/861/influenza+outbreak+event+prediction+via+twitter:

Info given on UC Irvine verbatim: 

"By identifying influenza-related tweets, the goal is to forecast the spatiotemporal patterns of influenza outbreaks for different locations and dates."

"The data is from the United States. The data comes from different states under different weeks. For each week, the task is to predict whether or not there is an influenza outbreak on the next date. More specifically, for influenza activity, there are four levels of flu activities from minimal to high according to CDC Flu Activity Map. An influenza outbreak occurrence is indicated if the activity level is high."

(see https://www.semanticscholar.org/paper/SimNest%3A-Social-Media-Nested-Epidemic-Simulation-Zhao-Chen/9f5ced0409dd869890cb152cb66e06539f95daa7)

Importing and loading the mat file we remove the headers that matlib uses and we don't need, then examine the shape of the remaining keys and their nested structure

In [53]:
from scipy.io import loadmat
import numpy as np

data = loadmat('./influenza_outbreak_dataset.mat')

# print(data.keys())
for key, val in data.items():
    if not key.startswith('__'):  # filter out metadata keys, these are generated by matlab and dont contain useful info for us
        print(f"Variable: {key} | Type: {type(val)} | Shape: {getattr(val, 'shape', 'N/A')}")

# examining shape of training/validation data
X_tr_0 = data['flu_X_tr'][0, 0]
Y_tr_0 = data['flu_Y_tr'][0, 0]

X_te_0 = data['flu_X_te'][0, 0]
Y_te_0 = data['flu_Y_te'][0, 0]

print("\nTest Features (X_te) shape:", X_te_0.shape)  
# not sure what these last 20 features are... something about CDC and coord related variables from SimNest paper
print("\nTest Features (X_te) last 20 features:", X_te_0[1, -20:].toarray())  

print("Test Target (Y_te) shape:", Y_te_0.shape)   

print("\nTrain Features (X_tr) shape:", X_tr_0.shape)  
print("Train Target (Y_tr) shape:", Y_tr_0.shape)  


Variable: flu_X_tr | Type: <class 'numpy.ndarray'> | Shape: (1, 48)
Variable: flu_Y_tr | Type: <class 'numpy.ndarray'> | Shape: (1, 48)
Variable: flu_X_te | Type: <class 'numpy.ndarray'> | Shape: (1, 48)
Variable: flu_Y_te | Type: <class 'numpy.ndarray'> | Shape: (1, 48)
Variable: flu_locs | Type: <class 'numpy.ndarray'> | Shape: (1, 48)
Variable: flu_keywords | Type: <class 'numpy.ndarray'> | Shape: (1, 525)

Test Features (X_te) shape: (485, 545)

Test Features (X_te) last 20 features: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
Test Target (Y_te) shape: (485, 1)

Train Features (X_tr) shape: (1095, 545)
Train Target (Y_tr) shape: (1095, 1)


As we can see our variables are tracking across 48 states, and have been stored as arrays in the matrix. 

Additionally, the data from over 3 years (1095 days) of observations have already been seperated in _tr for training and _te for testing. The existing split is 70/30. flu_locs simply stores state names. 

The keywords were used to identify tweets for a state in a given week that qualify as a record, these make up 525 of our features and the additional 20 are ...

The idea is to predict based on keyword frequencies on a weekly basis how likely it is for a flu outbreak to occur. 

In [39]:
# Decoding location names and keywords
loc_names = [str(x[0]) if isinstance(x, np.ndarray) else str(x) for x in data['flu_locs'].ravel()]
keywords = [str(w[0]) if isinstance(w, np.ndarray) else str(w) for w in data['flu_keywords'].ravel()]

print(f"First 5 regions: {loc_names[:5]}")
print(f"Total regions: {len(loc_names)}, Total keywords: {len(keywords)}")

# Extract features and targets for region 0
X_region_0 = data['flu_X_tr'][0, 0]
Y_region_0 = data['flu_Y_tr'][0, 0]

# If X is sparse, convert to dense for inspection; 
# if dense, check shape
# They're all increcibly sparse b/c a tweet will only have a few of the many keywords
if hasattr(X_region_0, 'toarray'):
    X_region_0 = X_region_0.toarray()

print("Region 0 Features (X) shape:", X_region_0.shape)  # typically (T_weeks, 525)
print("Region 0 Target (Y) shape:", Y_region_0.shape)      # typically (T_weeks, 1)

First 5 regions: ['wyoming', 'colorado', 'nebraska', 'washington', 'rhode island']
Total regions: 48, Total keywords: 525
Region 0 Features (X) shape: (1095, 545)
Region 0 Target (Y) shape: (1095, 1)


# Algorithm 1 Naive Bayes

We'll take the location list and find NY to run our first algorithm on. 

In [40]:
ny_idx = next(i for i, name in enumerate(loc_names) if 'new york' in name or name == 'ny')
print(f"New York index: {ny_idx}")  

New York index: 17


With this, we can extract the neccecary training and validation data from the matrix for NY state

In [ ]:
import pandas as pd
ny_idx = 17

# our data needs to be converted into an non-sparse version to be compatiable with our methods
# and only new york for training first model
def prep(mat):
    arr = mat[0, ny_idx]
    return arr.toarray() if hasattr(arr, 'toarray') else np.asarray(arr)

# No scaling required for GaussianNB — raw continuous values are preserved
#in this case these are the relative frequencies
X_train = pd.DataFrame(prep(data['flu_X_tr'])[:, :525])
# if we don't use ravel to flatten here model function will be upset, needs 1-d array
y_train = pd.Series( prep(data['flu_Y_tr']).ravel() > 0.0).astype(int)

X_test = pd.DataFrame(prep(data['flu_X_te'])[:, :525])
y_test = pd.Series((prep(data['flu_Y_te']).ravel() > 0.0)).astype(int)



print("X Train DataFrame: ", X_train.sample(5))
print("Y Train Series: ", y_train.sample(5))

print("X Test DataFrame: ", X_test.sample(5))
print("Y Test Series: ", y_test.sample(5))



X Train DataFrame:         0     1     2     3     4     5     6     7     8     9    ...  515  \
635    3.0   4.0   5.0   6.0   6.0   9.0   9.0  10.0  10.0  10.0  ...  0.0   
775    4.0   7.0   9.0   9.0  11.0  14.0  17.0  18.0  19.0  19.0  ...  0.0   
852   11.0  13.0  17.0  18.0  19.0  21.0  21.0  21.0  21.0  22.0  ...  0.0   
697   15.0  26.0  40.0  47.0  50.0  50.0  51.0  52.0  60.0  60.0  ...  1.0   
1014   0.0   1.0   1.0   2.0   9.0   9.0  10.0  10.0  10.0  10.0  ...  0.0   

      516  517  518  519  520  521  522  523  524  
635   0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  
775   0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  
852   0.0  0.0  0.0  0.0  0.0  2.0  0.0  0.0  0.0  
697   0.0  0.0  1.0  0.0  0.0  0.0  0.0  0.0  0.0  
1014  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  

[5 rows x 525 columns]
Y Train DataFrame:  830    0
991    0
85     0
409    0
80     0
dtype: int64
X Test DataFrame:       0    1    2    3    4     5     6     7     8     9    ...  515  516  \
3

We see the targets are a binary 1 or 0 indicating an outbreak or not, and the training samples represent the relative frequency for the keywords for that day in that state. Taking a look at the class distributions,

In [ ]:
# Check class distributions
print("Class Distributions")

print("y_train uniques:", np.unique(y_train))
print(f"y_train counts: Inactive={np.sum(y_train == 0)}, Active={np.sum(y_train == 1)}")

print("\ny_test uniques:", np.unique(y_train))
print(f"y_test counts:  Inactive={np.sum(y_test == 0)}, Active={np.sum(y_test == 1)}")

Class Distributions
y_train uniques: [0 1]
y_train counts: Inactive=1081, Active=14
y_test uniques: [0 1]
y_test counts:  Inactive=429, Active=56


I suspect the class imbalance will likely affect results here. 

In [47]:
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.naive_bayes import GaussianNB

# Fit Gaussian Naive Bayes
gnb = GaussianNB()
gnb.fit(X_train, y_train)

# Predict and evaluate
y_pred = gnb.predict(X_test)

print("\nGaussianNB - NY Test Set")
print(classification_report(y_test, y_pred, target_names=['Inactive (0)', 'Active Flu (1)'], zero_division=0))

print("Confusion Matrix")
print(confusion_matrix(y_test, y_pred))


GaussianNB - NY Test Set
                precision    recall  f1-score   support

  Inactive (0)       0.88      1.00      0.94       429
Active Flu (1)       0.00      0.00      0.00        56

      accuracy                           0.88       485
     macro avg       0.44      0.50      0.47       485
  weighted avg       0.78      0.88      0.83       485

Confusion Matrix
[[429   0]
 [ 56   0]]


The model is predicting 0 for all states due to how rare the success class is in the data, reducing the amount of features (choosing the top 100 from the 525) could improve this. Implementing fixed 50/50 priors and var smoothing did not improve the model's prediction ability.

# Algorithm 2